In [1]:
import openpyxl
from openpyxl import load_workbook, Workbook
import xlrd
from xlutils.copy import copy
import pprint

In [2]:
# Handling the Modern .xlsx File

print(" Processing Using openpyxl")

# 1. Load the Workbook
xlsx_file = "sample1.xlsx"
wb = load_workbook(xlsx_file)
sheet = wb.active 

# 2. Extract Data (Parsing)
sales_data = []

for row in sheet.iter_rows(min_row=2, values_only=True):
    # Mapping columns based on file structure:
    # Col 0: Postcode, 1: ID, 2: Name, 3: Year, 4: Value
    sales_rep = row[2] 
    value = row[4]
    
    if sales_rep and value:
        sales_data.append({
            "Sales_Rep": sales_rep,
            "Year": row[3],
            "Value": value
        })

# 3. Analysis: Sort Top 10 by Value (Replicating "Top 10 Child Labour" logic)
sales_data.sort(key=lambda x: x['Value'], reverse=True)

print("\nTop 10 High Value Transactions:")
for entry in sales_data[:10]:
    print(f"{entry['Sales_Rep']} ({entry['Year']}): {entry['Value']}")

# 4. Analysis: Filter Data (Replicating "Marriage > 40%" logic)
# Filtering for High Value transactions (> 50,000)
high_value_sales = [x for x in sales_data if x['Value'] > 50000]
print(f"\nNumber of transactions > 50,000: {len(high_value_sales)}")

# 5. Write to a New File (Output)
new_wb = Workbook()
new_ws = new_wb.active
new_ws.title = "Analysis Output"
new_ws.append(["Sales Rep", "Year", "Value"]) # Header

for item in high_value_sales:
    new_ws.append([item['Sales_Rep'], item['Year'], item['Value']])

new_wb.save("output_analysis_xlsx.xlsx")
print("Saved filtered data to 'output_analysis_xlsx.xlsx'")

# 6. Update Original File
# adding a "Processed" note in Column F for the first 5 rows
for i in range(2, 7): # Rows 2 to 6
    sheet.cell(row=i, column=6).value = "Verified by Python"

wb.save("sample1_updated.xlsx")
print("Saved updated original file to 'sample1_updated.xlsx'")

 Processing Using openpyxl

Top 10 High Value Transactions:
Ashish (2011): 99878.48920854433
Jane (2011): 99865.21157191237
Ashish (2011): 99743.83114915698
Jane (2012): 99436.42409836838
Ashish (2011): 98780.16287428583
John (2012): 98510.59024280483
Jane (2011): 98235.47271052479
Ashish (2013): 98199.93399223124
Ashish (2011): 97167.78450655767
Jane (2011): 97072.38171596124

Number of transactions > 50,000: 187
Saved filtered data to 'output_analysis_xlsx.xlsx'
Saved updated original file to 'sample1_updated.xlsx'


In [3]:
# Handling the Legacy .xls File
print("\n\n Processing Using xlrd/xlutils")

# 1. Load the Workbook (Formatting info preserved)
xls_file = "sample1.xls"
book = xlrd.open_workbook(xls_file, formatting_info=True)
sheet_xls = book.sheet_by_index(0)

# 2. Extract Data
people_data = []

for i in range(1, sheet_xls.nrows):
    row = sheet_xls.row_values(i)
    
    # Mapping columns: 
    # 0: ID, 1: First, 2: Last, 3: Gender, 4: Country, 5: Age
    first_name = row[1]
    last_name = row[2]
    country = row[4]
    age = row[5]
    
    people_data.append({
        "Name": f"{first_name} {last_name}",
        "Country": country,
        "Age": age
    })

# 3. Analysis: Sort by Age (Replicating logic)
people_data.sort(key=lambda x: x['Age'], reverse=True)

print("\nTop 5 Oldest People:")
for p in people_data[:5]:
    print(f"{p['Name']} ({p['Country']}): {p['Age']}")

# 4. Analysis: Filter Age > 30
adults = [p for p in people_data if p['Age'] > 30]
print(f"\nNumber of people over 30: {len(adults)}")

# 5. Update Original File
wb_copy = copy(book)
ws_copy = wb_copy.get_sheet(0)

# Write a note in the first data row, Column H (Index 7)
# (Original file has columns 0-7, so we write to column 8)
row_to_edit = 1
col_to_edit = 8
ws_copy.write(row_to_edit, col_to_edit, "Parsed by Python (xlutils)")

wb_copy.save("sample1_updated.xls")
print("Saved updated legacy file to 'sample1_updated.xls'")



 Processing Using xlrd/xlutils

Top 5 Oldest People:
Nereida Magwood (United States): 58.0
Etta Hurn (Great Britain): 56.0
Vincenza Weiland (United States): 40.0
Philip Gent (France): 36.0
Dulce Abril (United States): 32.0

Number of people over 30: 5
Saved updated legacy file to 'sample1_updated.xls'
